# HMLN train

Set paths and samples in `config.yaml`, then run the cells in order.


## Settings


In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
                    if (path / "experiments").is_dir() and (path / "lrpairs").is_dir())
sys.path.insert(0, str(PROJECT_ROOT / "experiments"))

from _shared.runtime import training_epochs, training_routes, scanvi_train_kwargs
from pathlib import Path
import yaml

DATASET = 'HMLN'
CWD = Path.cwd().resolve()
EXPERIMENT_DIR = CWD if (CWD / "config.yaml").is_file() else CWD / "experiments" / DATASET
REPO_ROOT = EXPERIMENT_DIR.parents[1]
import sys

CONFIG = yaml.safe_load(
    (EXPERIMENT_DIR / "config.yaml").read_text(encoding="utf-8")
)

def experiment_path(value):
    path = Path(value)
    return path if path.is_absolute() else (EXPERIMENT_DIR / path).resolve()

DATA_ROOT = experiment_path(CONFIG["data_root"])
RUN_ROOT = experiment_path(CONFIG["run_root"])
CHECKPOINT_ROOT = experiment_path(CONFIG["checkpoint_root"])
SCANVI_DIR = experiment_path(CONFIG["scanvi_dir"])
SCANVI_ADATA = SCANVI_DIR / "adata.h5ad"
STAGE1_CHECKPOINT_ROOT = experiment_path(CONFIG["stage1_checkpoint"])
STAGE2_CHECKPOINT_ROOT = experiment_path(CONFIG["stage2_checkpoint"])
LR_PAIRS = experiment_path(CONFIG["lr_pairs_path"])
TRANSITION_PRIOR = experiment_path(CONFIG["transition_prior_path"]) if "transition_prior_path" in CONFIG else None
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)



In [3]:
import re
import scanpy as sc
import torch
import numpy as np
import pandas as pd
import sys

import scvi
from pathlib import Path
import importlib
from stvirtual.models import stage1_3d as s1
from stvirtual.models import stage2_3d as s2


In [4]:
data_path = str(DATA_ROOT)
ckpt_path = str(CHECKPOINT_ROOT)
resl_path = str(RUN_ROOT)
lrpr_path = str(LR_PAIRS)


In [5]:
routes = training_routes(CONFIG["routes"])
fracs = [f"{src}_to_{tgt}" for src, tgt in routes]
seg_key = fracs[0]
steps = int(CONFIG["steps"])


In [6]:
adata = sc.read_h5ad(SCANVI_ADATA)
adata


AnnData object with n_obs × n_vars = 185673 × 28943
    obs: 'cell_ID_mask', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_reads', 'reads_per_counts', 'n_joined', 'exact_entropy', 'theoretical_entropy', 'exact_compression', 'theoretical_compression', 'n_counts', 'annotation', 'annotation_key', 'n_section', 'original_id_cleaned', 'sample_original', 'sample', '_scvi_batch', '_scvi_labels', 'cx_aligned', 'cy_aligned'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'annotation_colors', 'hvg', 'log1p', 'neighbors', 'pca', 'sample_colors', 'sample_mapping', 'umap'
    obsm: 'X_pca', 'X_scanVI', 'X_umap', 'spatial', 'spatial_3d_aligned'
    varm: 'PCs'
    layers: 'counts', 'raw'
    obsp: 'connectivities', 'distances'

In [7]:
adata.obs['sample'].unique()


['S1', 'S2', 'S3']
Categories (3, object): ['S1', 'S2', 'S3']

In [8]:
model = None


## Train Stage 1


In [9]:
res1 = {}
for route in routes:
    importlib.reload(s1)
    
    res1.update(s1.train_model_multislice(
        model=model,                  
        adata_all=adata,
        slice_key="sample",
        route_ids=route,
        save_root=str(STAGE1_CHECKPOINT_ROOT),
        x_key="cx_aligned",
        y_key="cy_aligned",
        latent_key=CONFIG["latent_key"],    
        steps=steps,  
        guide_eps=0.05,
        uot_eps=CONFIG["stage1_loss"]["uot_eps"], uot_tau=CONFIG["stage1_loss"]["uot_tau"], uot_lam_x=CONFIG["stage1_loss"]["uot_lam_x"], uot_lam_f=CONFIG["stage1_loss"]["uot_lam_f"],
        guide_topk=512, guide_temp=0.5, guide_schedule="linear",
        terrain_gap=False,
        cell_type_key='annotation',
        lam_context=10.0,
        lam_residual=1.0,
        lam_vsmooth=0.1,  
        lam_uot=CONFIG["stage1_loss"]["lam_uot"],
        lib_layer='raw',              
        latent_dim=10,             
        epochs=training_epochs(100),
        device="cuda:0",
    ))



[MultiSlice] Train segment: S1_to_S3 (n_src=55400, n_tgt=77868)
  Global norm route_ids=['S1', 'S3']  latent_dim=10

Pre-computing neighbor context features...


[ctx] scale 1/1: sparse matmul ...  (E=886378)


[ctx] scale 1/1: sparse matmul ...  (E=1245787)


Train (NeuralODE dopri5):   0%|          | 0/100 [00:00<?, ?it/s]

Train (NeuralODE dopri5):   0%|          | 0/100 [00:02<?, ?it/s, loss=0.0326, uot=0.0271, vs=0.0549]

...

Train (NeuralODE dopri5):  98%|█████████▊| 98/100 [03:51<00:04,  2.33s/it, loss=0.0238, uot=0.0167, vs=0.0535]

Train (NeuralODE dopri5):  99%|█████████▉| 99/100 [03:51<00:02,  2.33s/it, loss=0.0238, uot=0.0167, vs=0.0535]

Train (NeuralODE dopri5):  99%|█████████▉| 99/100 [03:53<00:02,  2.33s/it, loss=0.0240, uot=0.0167, vs=0.0554]

Train (NeuralODE dopri5): 100%|██████████| 100/100 [03:53<00:00,  2.33s/it, loss=0.0240, uot=0.0167, vs=0.0554]

Train (NeuralODE dopri5): 100%|██████████| 100/100 [03:53<00:00,  2.34s/it, loss=0.0240, uot=0.0167, vs=0.0554]

In [10]:
res1['S1_to_S3'].keys()


dict_keys(['guide_fn', 'guide_cache', 'net', 'func', 'coords0', 'Z0', 'lib0', 'coords_tgt', 'Zt', 'lib_tgt', 'best', 'norm_stats'])

## Create boundaries


In [11]:
from stvirtual.utils import boundary_2d as boundary
from stvirtual.utils.stage1_results import save_res as save_stage1_results
from stvirtual.utils.trajectory import simulation_trace_from_out

boundary_cfg = CONFIG["boundary"]
saved = save_stage1_results(
    res1=res1, adata_all=adata,
    out_dir=str(CHECKPOINT_ROOT / "stage1_res"),
    slice_key="sample", ann_key="annotation",
    save_prefix="simulation_stage1", steps=steps,
    n_cache=boundary_cfg["n_cache"], unnormalize=False,
)

for stage_key in fracs:
    source_name = stage_key.split("_to_", 1)[0]
    match = re.search(r"\d+", source_name)
    source_number = int(match.group(0)) if match else 10**9
    trace = simulation_trace_from_out(res1[stage_key], steps=steps, n_cache=boundary_cfg["n_cache"], unnormalize=False)
    frames = [value.detach().cpu().numpy().astype(np.float32) for value in trace["x"]]
    stage_dir = RUN_ROOT / "bound" / stage_key
    stage_dir.mkdir(parents=True, exist_ok=True)
    statistics = []
    for frame_index, coordinates in enumerate(frames):
        shell, fraction, outside, expansion = boundary.make_shell_adaptive(
            coordinates, alpha_factor=boundary_cfg["alpha_factor"],
            seed=2026 + source_number + frame_index,
            n_resample=boundary_cfg["n_resample"], target_frac=boundary_cfg["target_frac"],
            expand0=boundary_cfg["expand0"], expand_step=boundary_cfg["expand_step"],
            expand_max=boundary_cfg["expand_max"], fallback_expand=boundary_cfg["fallback_expand"],
        )
        boundary.save_shell_csv(shell, stage_dir / f"bound_z{frame_index:03d}.csv")
        boundary.plot_shell_coverage(coordinates, shell, outside, stage_dir / f"bound_z{frame_index:03d}.png")
        statistics.append({"frame": frame_index, "n_cells": len(coordinates), "fraction_inside": fraction, "n_outside": len(outside), "expansion": expansion})
    pd.DataFrame(statistics).to_csv(stage_dir / "bound_stats.csv", index=False)

print("Stage-1 traces:", saved)
print("Boundary root:", RUN_ROOT / "bound")


Stage-1 traces: {'S1_to_S3': '/home/zhouyj/stVirtual_tutorial/experiments/HMLN/artifacts/checkpoints/stage1_res/simulation_stage1_S1_to_S3.npz'}
Boundary root: /home/zhouyj/stVirtual_tutorial/experiments/HMLN/artifacts/results/bound


## Train Stage 2


In [12]:
importlib.reload(s2)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

ctx = s2.build_global_ctx(
    adata_path=str(SCANVI_ADATA),
    lr_pairs_path=lrpr_path,
    ckpt_3dslice=str(STAGE1_CHECKPOINT_ROOT / "S1_to_S3" / "checkpoints" / "best.pt"),
    device=device,
    layer_col="annotation",
)

stages = []
for sk in fracs:
    src, tgt = sk.split("_to_", 1)
    stages.append(
        s2.StageCfg(
            src=src,
            tgt=tgt,
            out_npz_path=saved[sk],
            bound_dir=f"{resl_path}/bound/{sk}",
            decoder_checkpoint=str(experiment_path(CONFIG["decoder_checkpoint"].format(src=src, tgt=tgt))),
            scanvi_dir=None,
            model_type="scanvi",
            use_lr=True,
            lr_source="decoder",
            use_latent=True,
            latent_key=CONFIG["latent_key"],
            z_csv_offset=0,
            layer_col="annotation",
        )
    )

outs = s2.run_multi_stages(
    ctx=ctx,
    sample_key = 'sample',
    stages=stages,
    best_ckpt_dir=str(STAGE2_CHECKPOINT_ROOT),
    train_kwargs=dict(EPOCHS=training_epochs(100), LR=1e-4),
    rl_xy=CONFIG["stage2_loss"]["rl_xy"],
    rl_z=CONFIG["stage2_loss"]["rl_z"],
)

for o in outs:
    print(o["best_ckpt_path"])


[Grid] H=236, W=316, H×W=74576


[auto cap] 5 {'src': {'q': 0.9, 'quantile_value': 3.0, 'mean': 1.8551905155181885, 'max': 7.0, 'n_cells_used': 29853}, 'tgt': {'q': 0.9, 'quantile_value': 5.0, 'mean': 2.4625556468963623, 'max': 9.0, 'n_cells_used': 31313}, 'cap0': 3, 'capT': 5, 'cap': 5}


Training:   0%|          | 0/100 [00:00<?, ?it/s]

[KeOps] Generating code for Max_SumShiftExpWeight_Reduction reduction (with parameters 0) of formula [c-1/2*(d*Sum((a-b)**2)),1] with a=Var(0,2,0), b=Var(1,2,1), c=Var(2,1,1), d=Var(3,1,2) ... 

OK


...

Training:  99%|█████████▉| 99/100 [38:27<00:23, 23.16s/it, rew=-0.6435, best=-0.6385, tgt=0.5457, occ=0.9493]

Training:  99%|█████████▉| 99/100 [38:48<00:23, 23.16s/it, rew=-0.6417, best=-0.6385, tgt=0.5438, occ=0.9499]

Training: 100%|██████████| 100/100 [38:48<00:00, 22.74s/it, rew=-0.6417, best=-0.6385, tgt=0.5438, occ=0.9499]

Training: 100%|██████████| 100/100 [38:48<00:00, 23.29s/it, rew=-0.6417, best=-0.6385, tgt=0.5438, occ=0.9499]

/home/zhouyj/stVirtual_tutorial/experiments/HMLN/artifacts/checkpoints/stage2/policy_S1_to_S3.pt


In [13]:
ckpt_map = {}
for o in outs:
    cfg = o["stage"]
    ckpt_map[f"{cfg.src}_to_{cfg.tgt}"] = o["best_ckpt_path"]

ckpt_map


{'S1_to_S3': '/home/zhouyj/stVirtual_tutorial/experiments/HMLN/artifacts/checkpoints/stage2/policy_S1_to_S3.pt'}

## Run simulation


In [14]:
importlib.reload(s2)
simulations = {}
for cfg in stages:
    seg_key = f"{cfg.src}_to_{cfg.tgt}"
    simulations[seg_key] = s2.simulation_policy_one_stage(
        s2, ctx, cfg, ckpt_map[seg_key],
        seed=2026, ADVECT_LATENT=True,
        output_dir=Path(resl_path) / 'simulation' / seg_key, output_prefix=seg_key,
    )
    print(seg_key, "Tp1=", len(simulations[seg_key]["coords"]))


[Grid] H=236, W=316, H×W=74576


[auto cap] 5 {'src': {'q': 0.9, 'quantile_value': 3.0, 'mean': 1.8551905155181885, 'max': 7.0, 'n_cells_used': 29853}, 'tgt': {'q': 0.9, 'quantile_value': 5.0, 'mean': 2.4625556468963623, 'max': 9.0, 'n_cells_used': 31313}, 'cap0': 3, 'capT': 5, 'cap': 5}


S1_to_S3 Tp1= 11


In [15]:
SIMULATION_OBS_FIELDS = ("uid", "parent_uid")

def persist_simulation_obs(simulation_result):
    output_paths = simulation_result.get("output_paths")
    if not isinstance(output_paths, list):
        raise KeyError("simulation_result has no output_paths; pass output_dir to simulation_policy_one_stage")
    if len(output_paths) != len(simulation_result["coords"]):
        raise ValueError("output_paths and simulation frames have different lengths")

    for frame_index, output_path in enumerate(output_paths):
        frame_adata = sc.read_h5ad(output_path)
        for field in SIMULATION_OBS_FIELDS:
            frames = simulation_result.get(field)
            if not isinstance(frames, list):
                raise KeyError(f"simulation_result does not provide {field!r}")
            values = np.asarray(frames[frame_index])
            if values.ndim != 1 or len(values) != frame_adata.n_obs:
                raise ValueError(
                    f"{field} at frame {frame_index} has shape {values.shape}; "
                    f"expected ({frame_adata.n_obs},)"
                )
            frame_adata.obs[field] = values
        frame_adata.obs_names = frame_adata.obs["uid"].astype(str).to_numpy()
        frame_adata.write_h5ad(output_path, compression="gzip")

    return {"frames_updated": len(output_paths), "obs_fields": list(SIMULATION_OBS_FIELDS)}

simulation_obs_reports = {
    route: persist_simulation_obs(route_simulation)
    for route, route_simulation in simulations.items()
}
simulation_obs_reports


{'S1_to_S3': {'frames_updated': 11, 'obs_fields': ['uid', 'parent_uid']}}

In [16]:
ro = simulations[seg_key]

for t in range(len(ro["coords"])):
    N  = int(np.asarray(ro["coords"][t]).shape[0])
    nb = int(np.asarray(ro["is_birth"][t]).sum())  
    print(f"{seg_key} | t={t:02d} | N={N} | new_birth={nb}")


S1_to_S3 | t=00 | N=55400 | new_birth=0
S1_to_S3 | t=01 | N=57642 | new_birth=10028
S1_to_S3 | t=02 | N=59883 | new_birth=10209
S1_to_S3 | t=03 | N=62125 | new_birth=10853
S1_to_S3 | t=04 | N=64368 | new_birth=11690
S1_to_S3 | t=05 | N=66611 | new_birth=12818
S1_to_S3 | t=06 | N=68855 | new_birth=13990
S1_to_S3 | t=07 | N=71098 | new_birth=15451
S1_to_S3 | t=08 | N=73345 | new_birth=16811
S1_to_S3 | t=09 | N=75593 | new_birth=17261
S1_to_S3 | t=10 | N=77868 | new_birth=17742


In [17]:
from _shared.runtime import save_run_gallery

overview_path = save_run_gallery(EXPERIMENT_DIR)


Saved 12 panels: /home/zhouyj/stVirtual_tutorial/experiments/HMLN/artifacts/results/overview/S1_to_S3.png
